# 01: Engenharia de Dados e Modelagem de Performance
Nesta etapa, consolidamos a 'Tabela Mestra de Performance' através do Join real entre o histórico de disparos e a dimensão de usuários. 

**Diferenciais de Senioridade:**
- Tipagem segura para chaves hasheadas (String).
- Desestruturação de arrays JSON para granularidade de sistema.
- Gestão de memória para processamento em larga escala.

In [1]:
import pandas as pd
import gc

# 1. Carga de Dados via engine PyArrow
print(">>> Iniciando ingestão de fontes brutas...")
df_disparo = pd.read_parquet('../data/whatsapp_base_disparo_mascarado')
df_dim = pd.read_parquet('../data/whatsapp_dim_telefone_mascarado')

# PADRONIZAÇÃO DE TIPOS: Utilizamos strings para identificadores e telefones.
# Esta prática evita a perda de precisão ou remoção acidental de zeros à esquerda (leading zeros) em processos de merge.
df_disparo['contato_telefone'] = df_disparo['contato_telefone'].astype(str)
df_dim['telefone_numero'] = df_dim['telefone_numero'].astype(str)
df_dim['telefone_ddd'] = df_dim['telefone_ddd'].astype(str)

print(f"Ingestão concluída. Disparos: {df_disparo.shape} | Dimensão: {df_dim.shape}")

>>> Iniciando ingestão de fontes brutas...


Ingestão concluída. Disparos: (392921, 16) | Dimensão: (283289, 11)


In [2]:
print(">>> Desestruturando origens e normalizando base dimensional...")
# DESESTRUTURAÇÃO: Expansão da coluna 'telefone_aparicoes' para permitir análise por sistema de origem.
df_dim_exploded = df_dim.explode('telefone_aparicoes')
df_fontes = pd.json_normalize(df_dim_exploded['telefone_aparicoes'])
df_fontes.index = df_dim_exploded.index

# Consolidação da base enriquecida com metadados de sistema e proprietário
df_dim_full = pd.concat([df_dim_exploded.drop(columns=['telefone_aparicoes']), df_fontes], axis=1)

# JOIN ESTRATÉGICO: Cruzamento real entre histórico de entrega e cadastro RMI.
# O resultado é a base de evidências para o cálculo das taxas empíricas de sucesso.
df_join = pd.merge(
    df_disparo,
    df_dim_full,
    left_on='contato_telefone',
    right_on='telefone_numero',
    how='inner'
)

print(f"Tabela Mestra de Performance criada com {df_join.shape[0]} registros.")

>>> Desestruturando origens e normalizando base dimensional...


Tabela Mestra de Performance criada com 6294306 registros.


In [3]:
# GESTÃO DE MEMÓRIA: Limpeza de objetos temporários e coleta de lixo (GC).
# Essencial para garantir a escalabilidade do notebook em ambientes de produção com milhões de registros.
df_join.to_parquet('../data/base_performance_mestra.parquet')
del df_disparo, df_dim, df_dim_full
gc.collect()
print("Persistência concluída. Ambiente otimizado.")

Persistência concluída. Ambiente otimizado.
